# Visualisations

Loads saved results from `Results/` and produces publication-quality figures.
Each figure is saved to `Plots/` as a PDF (+ PNG) with metadata embedded in
a sidecar `_meta.json` file.

**Run order**
1. *Setup* — imports, style, helpers
2. *Q(k) diagnostics* — raw and interpolated self-energy
3. *Exciton & photon dispersion*
4. *Lower polariton dispersion*
5. *Hopfield coefficients*
6. *Detuning*
7. *Interaction strengths vs k*


In [ ]:
import sys
sys.path.insert(0, '.')

import json
import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RectBivariateSpline, PchipInterpolator

from polaritons.io         import load_result, list_results, load_latest
from polaritons.dispersion import DispersionModel
from polaritons.many_body  import (
    hopfield_coefficients,
    polariton_interaction_strength,
)
from polaritons.parameters import Params

# ---------------------------------------------------------------------------
# Matplotlib style
# ---------------------------------------------------------------------------
plt.rcParams.update({
    'font.family'               : 'serif',
    'font.serif'                : ['DejaVu Serif'],
    'font.size'                 : 14,
    'axes.labelsize'            : 16,
    'axes.titlesize'            : 16,
    'xtick.labelsize'           : 14,
    'ytick.labelsize'           : 14,
    'legend.fontsize'           : 13,
    'figure.titlesize'          : 18,
    'axes.formatter.limits'     : (-99, 99),
})

PLOTS_DIR = Path("Plots")
PLOTS_DIR.mkdir(exist_ok=True)

# ---------------------------------------------------------------------------
# Figure-saving helper
# ---------------------------------------------------------------------------

def save_figure(fig: plt.Figure, name: str, meta: dict | None = None,
                dpi: int = 150) -> None:
    """
    Save `fig` to Plots/<name>.pdf and Plots/<name>.png, and write metadata
    to Plots/<name>_meta.json.

    Parameters
    ----------
    fig  : matplotlib Figure
    name : file stem (no extension)
    meta : extra key-value pairs to record in the sidecar JSON
    dpi  : raster DPI for the PNG copy
    """
    pdf_path  = PLOTS_DIR / f"{name}.pdf"
    png_path  = PLOTS_DIR / f"{name}.png"
    json_path = PLOTS_DIR / f"{name}_meta.json"

    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, bbox_inches="tight", dpi=dpi)

    record = {
        "saved_at" : datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "pdf"      : str(pdf_path),
        "png"      : str(png_path),
        **(meta or {}),
    }
    with open(json_path, "w") as f:
        json.dump(record, f, indent=2, default=str)
    print(f"Saved  {pdf_path.name}")


print("Setup complete.")


## Load results

Choose which saved result set to visualise.  By default the most recently
saved Q result is loaded; change `Q_STEM` / `SWEEP_IDX` to pick a specific run.


In [ ]:
# Set Q_STEM = None to auto-load the most recent result
Q_STEM = None

if Q_STEM is None:
    Q_results, q_meta = load_latest("Results/Q_results", prefix="Q")
else:
    Q_results, q_meta = load_result("Results/Q_results", Q_STEM)

print(f"Loaded Q_results  shape={Q_results.shape}  dtype={Q_results.dtype}")
print(f"  saved_at : {q_meta['saved_at']}")
print(f"  xi       : {q_meta['params']['xi']:.2e} (natural units)")

# Reconstruct Params from metadata
raw = q_meta["params"]
p_si = Params(
    E_bind       = raw["E_bind"],
    E_gap_bare   = raw["E_gap_bare"],
    m_e          = raw["m_e"],
    m_h          = raw["m_h"],
    m_rest       = raw["m_rest"],
    Omega        = raw["Omega"],
    m_prime      = raw["m_prime"],
    n_refr       = raw.get("n_refr", 3.0),
    N_qw         = raw.get("N_qw", 1),
    D_0          = raw["D_0"],
    xi           = raw["xi"],
    T            = raw["T"],
    concentration = raw["concentration"],
    g_ex         = raw["g_ex"],
)
p_nat = p_si.to_natural()

eta_grid  = np.array(q_meta["eta_grid"])
q_picard  = np.array(q_meta["q_picard"])   # natural units

# Build dispersion model
model = DispersionModel(p_si, q_picard, eta_grid, Q_results)

# Momentum grid for plotting (SI, m^-1)
k_disp = np.linspace(0, 4e6, 200)    # standard range
k_log  = np.linspace(0, 4e7, 500)    # extended range

print(f"\nParams: E_bind={p_si.E_bind*1e3:.1f} meV  Omega={p_si.Omega*1e3:.1f} meV  "
      f"T={p_si.T:.0f} K  xi={p_si.xi*1e9:.0f} nm")


## Q(k) diagnostics

Raw and interpolated self-energy for selected disorder values.


In [ ]:
plot_eta_idx = [0, 5, 10, 15, 20]

fig, axs = plt.subplots(1, 2, figsize=(18, 6))

for idx in plot_eta_idx:
    clr = plt.cm.magma(idx / len(eta_grid))
    lbl = f"η={eta_grid[idx]:.2f}"
    axs[0].plot(q_picard / p_si.L_unit * 1e-6, Q_results[idx].real,
                color=clr, label=f"{lbl} (Re)")
    axs[0].plot(q_picard / p_si.L_unit * 1e-6, Q_results[idx].imag,
                color=clr, linestyle='--', label=f"{lbl} (Im)")

    k_si_test = np.linspace(0, q_picard.max() / p_si.L_unit * 0.9, 300)
    Q_interp  = model.Q(k_si_test, eta_grid[idx])
    axs[1].plot(k_si_test * 1e-6, np.real(Q_interp) / p_si.E_unit,
                color=clr, label=f"{lbl} (Re)")
    axs[1].plot(k_si_test * 1e-6, np.imag(Q_interp) / p_si.E_unit,
                color=clr, linestyle='--', label=f"{lbl} (Im)")

for ax, title in zip(axs, ["Raw Q(k)  (natural units)", "Interpolated Q(k)  (natural units)"]):
    ax.set_xlabel("k (µm⁻¹)")
    ax.set_ylabel("Q")
    ax.set_title(title)
    ax.legend(ncol=2, fontsize=10)
    ax.grid(alpha=0.3)

plt.tight_layout()
save_figure(fig, "Q_diagnostics", meta={"eta_indices": plot_eta_idx, "source_stem": Q_STEM})
plt.show()


## Exciton energy at k=0 vs disorder strength


In [ ]:
E_ex_k0 = np.array([model.E_ex(np.array([0.0]), eta)[0] for eta in eta_grid])

eta_dense      = np.linspace(eta_grid[0], eta_grid[-1], 201)
E_real_dense   = PchipInterpolator(eta_grid, np.real(E_ex_k0))(eta_dense)
E_imag_dense   = PchipInterpolator(eta_grid, np.imag(E_ex_k0))(eta_dense)

fig, ax = plt.subplots(figsize=(8, 6))
ax_r = ax.twinx()

ax.plot(eta_dense,        E_real_dense,         color='r', lw=1.5, label="Real")
ax_r.plot(eta_dense, 1e3 * E_imag_dense,        color='b', lw=1.5, label="Imaginary")

ax.set_xlabel("Disorder strength η")
ax.set_ylabel(r"Re[$E_x$(0)] (eV)",  color='r')
ax_r.set_ylabel(r"Im[$E_x$(0)] (meV)", color='b')
ax.set_title("Exciton energy at k=0 vs disorder strength")
ax.grid(alpha=0.3)
ax.legend(loc="upper left"); ax_r.legend(loc="upper right")
plt.tight_layout()
save_figure(fig, "Eex_k0_vs_eta")
plt.show()


## Dispersion relations

Exciton and photon dispersion for a selected disorder value, and full
exciton / LP dispersions for all η.


In [ ]:
eta_probe   = eta_grid[10]
k_cm        = 0.01 * k_disp       # m^-1 → cm^-1

E_ex_probe  = model.E_ex(k_disp, eta_probe)
E_ex_clean  = model.E_ex(k_disp, 0.0)
E_ph_probe  = model.E_ph(k_disp, eta_probe)
E_ph_clean  = model.E_ph_untuned(k_disp)

for suffix, E_ph_use in [
    ("disorder-tuned cavity",  E_ph_probe),
    ("disorder-free cavity",   E_ph_clean),
]:
    fig, axs = plt.subplots(1, 2, figsize=(18, 7))
    axs[0].plot(k_cm, np.real(E_ex_probe), color='r',     lw=2, label=f"Exciton η={eta_probe:.1f}")
    axs[0].plot(k_cm, np.real(E_ex_clean), color='b',     lw=2, label="Exciton η=0")
    axs[0].plot(k_cm, E_ph_use,            color='green', lw=2, label="Photon")
    axs[1].plot(k_cm, 1e3*np.imag(E_ex_probe), color='r', lw=2, label=f"Exciton η={eta_probe:.1f}")
    axs[1].plot(k_cm, 1e3*np.imag(E_ex_clean), color='b', lw=2, label="Exciton η=0")
    for ax in axs:
        ax.set_xlabel("k (cm⁻¹)")
        ax.legend(loc="lower right")
        ax.grid(alpha=0.3)
        ax.xaxis.set_ticks(np.linspace(0, 40_000, 9))
        ax.set_xticklabels([f"{int(x):,}" for x in ax.get_xticks()])
    axs[0].set_ylabel("E (eV)")
    axs[1].set_ylabel("E (meV)")
    axs[0].set_title(f"Dispersion — Real  [{suffix}]")
    axs[1].set_title(f"Dispersion — Imaginary  [{suffix}]")
    axs[0].text(0.04, 0.92, "(a)", transform=axs[0].transAxes, fontsize=22)
    axs[1].text(0.04, 0.92, "(b)", transform=axs[1].transAxes, fontsize=22)
    plt.tight_layout()
    label = suffix.replace(" ", "_").replace("-", "")
    save_figure(fig, f"dispersion_{label}", meta={"eta_probe": eta_probe})
    plt.show()


## Lower polariton dispersion — all η


In [ ]:
legend_etas = [0.0, 0.5, 1.0, 1.5, 2.0]

for label, tuned in [("Disorder-tuned cavity", True), ("Disorder-free cavity", False)]:
    fig, axs = plt.subplots(1, 2, figsize=(18, 7))
    for ei, eta in enumerate(eta_grid):
        clr  = plt.cm.magma(ei / len(eta_grid))
        lbl  = f"η={eta:.1f}" if eta in legend_etas else "_nolegend_"
        E_lp = model.E_LP(k_disp, eta, disorder_tuned=tuned)
        axs[0].plot(0.01 * k_disp, np.real(E_lp), color=clr, label=lbl)
        axs[1].plot(0.01 * k_disp, 1e3 * np.imag(E_lp), color=clr, label=lbl)
    for ax in axs:
        ax.set_xlabel("k (cm⁻¹)")
        ax.legend(loc="lower right")
        ax.grid(alpha=0.3)
        ax.xaxis.set_ticks(np.linspace(0, 40_000, 9))
        ax.set_xticklabels([f"{int(x):,}" for x in ax.get_xticks()])
    axs[0].set_ylabel("Re[E_LP] (eV)")
    axs[1].set_ylabel("Im[E_LP] (meV)")
    axs[0].set_title(f"LP Dispersion — Real  [{label}]")
    axs[1].set_title(f"LP Dispersion — Imaginary  [{label}]")
    axs[0].text(0.04, 0.95, "(a)", transform=axs[0].transAxes, fontsize=22)
    axs[1].text(0.04, 0.95, "(b)", transform=axs[1].transAxes, fontsize=22)
    plt.tight_layout()
    fname = "LP_dispersion_" + ("tuned" if tuned else "untuned")
    save_figure(fig, fname)
    plt.show()


## Hopfield coefficients


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(16, 6))
for eta in legend_etas:
    clr  = plt.cm.magma(int(eta / 2.0 * 255))
    E_lp = model.E_LP(k_disp, eta, disorder_tuned=True)
    X_LP, C_LP = hopfield_coefficients(model, eta, k_disp, E_lp)
    axs[0].plot(0.01 * k_disp, np.abs(X_LP)**2, color=clr, label=f"η={eta:.1f}")
    axs[1].plot(0.01 * k_disp, np.abs(C_LP)**2, color=clr, label=f"η={eta:.1f}")

for ax, title in zip(axs, ["Exciton fraction |X_LP|²", "Photon fraction |C_LP|²"]):
    ax.set_xlabel("k (cm⁻¹)")
    ax.set_ylabel("Hopfield coefficient²")
    ax.set_title(title)
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(alpha=0.3)
    ax.xaxis.set_ticks(np.linspace(0, 40_000, 9))
    ax.set_xticklabels([f"{int(x):,}" for x in ax.get_xticks()])

plt.tight_layout()
save_figure(fig, "hopfield_coefficients")
plt.show()


## Detuning Δ(k, η) = E_ph − E_ex


In [ ]:
k_wide = np.linspace(0, 1e10, 400)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

for ei, eta in enumerate(eta_grid):
    clr = plt.cm.magma(ei / len(eta_grid))
    lbl = fr"$\eta={eta:.1f}$" if eta in legend_etas else "_nolegend_"
    delta = model.E_ph(k_wide, eta) - model.E_ex(k_wide, eta)
    ax1.plot(k_wide * 1e-6, 1e3 * np.real(delta), color=clr, label=lbl)
    ax2.plot(k_wide * 1e-6, 1e3 * np.imag(delta), color=clr, label=lbl)

for ax in (ax1, ax2):
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_xlabel(r"$k$ (µm$^{-1}$)")
    ax.legend()
    ax.grid(alpha=0.3)

ax1.set_ylabel(r"Re[$E_{ph}-E_{ex}$] (meV)")
ax1.set_title("Real detuning vs momentum")
ax2.set_ylabel(r"Im[$E_{ph}-E_{ex}$] (meV)")
ax2.set_title("Imaginary detuning vs momentum")

plt.tight_layout()
save_figure(fig, "detuning")
plt.show()


## Polariton interaction strength vs k


In [ ]:
# Parameters for the many-body calculation
L_TERMS = 100
K_UPPER = 1e8    # m^-1
N_K     = 100_000

for tuned, cavity_label in [(True, "disorder_tuned"), (False, "disorder_free")]:
    for bare, g_label, y_label in [
        (True,  "bare",    "g (µeV·µm²)"),
        (False, "screened", "g' (µeV·µm²)"),
    ]:
        fig, ax = plt.subplots(figsize=(8, 6))
        for ei, eta in enumerate(eta_grid):
            clr = plt.cm.magma(ei / len(eta_grid))
            lbl = f"η={eta:.1f}" if eta in legend_etas else "_nolegend_"
            g = polariton_interaction_strength(
                model, eta, k_disp,
                bare=bare,
                L_terms=L_TERMS, k_upper=K_UPPER, n_k=N_K,
                disorder_tuned=tuned,
            )
            ax.plot(0.01 * k_disp, 1e18 * g, color=clr, label=lbl, alpha=0.9)
        ax.set_xlabel("k (cm⁻¹)")
        ax.set_ylabel(y_label)
        ax.set_title(f"{g_label.capitalize()} interaction strength  [{cavity_label.replace('_', ' ')}]")
        ax.legend()
        ax.set_ylim(0, None)
        ax.grid(alpha=0.3)
        ax.xaxis.set_ticks(np.linspace(0, 40_000, 5))
        ax.set_xticklabels([f"{int(x):,}" for x in ax.get_xticks()])
        plt.tight_layout()
        save_figure(fig, f"g_{g_label}_{cavity_label}",
                    meta={"bare": bare, "disorder_tuned": tuned,
                          "L_terms": L_TERMS, "k_upper": K_UPPER})
        plt.show()
